In [1]:
import pandas as pd
import db_mgmt as mgmt
import os
import fix_imports as fi
import shutil

In [44]:
data_schema = 'dbs/canoe_dataset_schema 5.sql'
db_file = 'dbs/canoe_fuel.sqlite'
os.remove(db_file) if os.path.exists(db_file) else None
mgmt.convert_sql_to_sqlite(data_schema, db_file)
new_transport = mgmt.sqlite_to_dfs(db_file)

dict_keys(['MetaData', 'MetaDataReal', 'SeasonLabel', 'SectorLabel', 'CapacityCredit', 'CapacityFactorProcess', 'CapacityFactorTech', 'CapacityToActivity', 'CommodityLabel', 'Commodity', 'CommodityType', 'ConstructionInput', 'CostEmission', 'CostFixed', 'CostInvest', 'CostVariable', 'Demand', 'DemandSpecificDistribution', 'EndOfLifeOutput', 'Efficiency', 'EfficiencyVariable', 'EmissionActivity', 'EmissionEmbodied', 'EmissionEndOfLife', 'ExistingCapacity', 'TechGroup', 'LoanLifetimeProcess', 'LoanRate', 'LifetimeProcess', 'LifetimeTech', 'Operator', 'LimitGrowthCapacity', 'LimitDegrowthCapacity', 'LimitGrowthNewCapacity', 'LimitDegrowthNewCapacity', 'LimitGrowthNewCapacityDelta', 'LimitDegrowthNewCapacityDelta', 'LimitStorageLevelFraction', 'LimitActivity', 'LimitActivityShare', 'LimitAnnualCapacityFactor', 'LimitCapacity', 'LimitCapacityShare', 'LimitNewCapacity', 'LimitNewCapacityShare', 'LimitResource', 'LimitSeasonalCapacityFactor', 'LimitTechInputSplit', 'LimitTechInputSplitAnnual'

In [6]:
db_core = 'dbs/residential/fuel.sqlite'
db_path = f'dbs/canoe_fuel.sqlite'

os.remove(db_path) if os.path.exists(db_path) else None
shutil.copy2(db_core, db_path)



'dbs/canoe_fuel.sqlite'

In [7]:


data = mgmt.sqlite_to_dfs(db_path)

df = data['CostVariable']

dsl = df.loc[(df['tech'].str.contains('dsl', case=False))&(df['region']=='ON')&(df['period']==2025)]
print(dsl)

    region  period      tech  vintage       cost       units  \
240     ON    2025   F_E_DSL     2025  25.029401  2020 M$/PJ   
265     ON    2025   F_I_DSL     2025  24.814443  2020 M$/PJ   
271     ON    2025   F_T_DSL     2025  29.099577  2020 M$/PJ   
272     ON    2025  F_T_RDSL     2025  34.286608  2020 M$/PJ   
279     ON    2025   F_A_DSL     2025  29.099577  2020 M$/PJ   

                notes data_source  dq_cred  dq_geog  dq_struc  dq_tech  \
240            diesel          F1        2        3         2        1   
265            diesel          F1        2        3         2        1   
271            diesel          F1        2        3         2        1   
272  renewable diesel          F1        2        3         2        1   
279            diesel          F1        2        3         2        1   

     dq_time      data_id  
240        1  FUELHRON002  
265        1  FUELHRON002  
271        1  FUELHRON002  
272        1  FUELHRON002  
279        1  FUELHRON002  


In [37]:
db_path = f'dbs/canoe_fuel.sqlite'
data_fuel = mgmt.sqlite_to_dfs(db_path)
cav = data_fuel['CostVariable']

In [27]:
"low".upper()

'LOW'

In [31]:
fuel_techs = pd.read_csv('dbs/tech_fuel.csv')
importing = pd.DataFrame()
for fuel in fuel_techs['fuel'].unique():
    print(fuel)
    df = cav.loc[cav['tech'].isin(fuel_techs[fuel_techs['fuel'] == fuel]['tech'])]
    # print(df)
    for region in df['region'].unique():
        for period in df['period'].unique():
            for vintage in df['vintage'].unique():
            # print(fuel, region, period)
                sub_df = df.loc[(df['region'] == region) & (df['period'] == period) & (df['vintage'] == vintage)]
                # print(sub_df)
                if len(sub_df) > 0: 
                    min_cost = sub_df['cost'].min()
                    sub_df = pd.concat([sub_df, sub_df.iloc[[0]]],ignore_index=True)
                    sub_df.loc[0,'tech'] = f'F_IMP_{fuel.upper()}'
                    sub_df.loc[0,'cost'] = min_cost
                    sub_df.loc[~sub_df['tech'].str.contains('IMP'), 'cost'] = sub_df.loc[~sub_df['tech'].str.contains('IMP'), 'cost'] - min_cost  
                    importing = pd.concat([importing, sub_df])
                    if (not sub_df['tech'].str.contains('IMP').any()) and (sub_df['cost'].sum()>0):
                        print(sub_df)



bio_g
bio_m
ng
coal
oil
dsl
u_nat
u_enr
gsl
elc
lpg
wood
bio
h2
hfo
ngl
oth
pcoke
coke
eth
rdsl
cng
jtf
spk
mdo
lng
prop


In [35]:
fuels_cost_var = importing.loc[importing['cost']!=0].copy()

importing[importing['tech'].str.contains('DSL', case=False)]


,region,period,tech,vintage,cost,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,AB,2025,F_IMP_DSL,2025,24.814443,2020 M$/PJ,diesel,F1,2,3,2,1,1,FUELHRAB002
1,AB,2025,F_I_DSL,2025,0.000000,2020 M$/PJ,diesel,F1,2,3,2,1,1,FUELHRAB002
2,AB,2025,F_T_DSL,2025,4.285134,2020 M$/PJ,diesel,F1,2,3,2,1,1,FUELHRAB002
3,AB,2025,F_A_DSL,2025,4.285134,2020 M$/PJ,diesel,F1,2,3,2,1,1,FUELHRAB002
4,AB,2025,F_E_DSL,2025,0.214958,2020 M$/PJ,diesel,F1,2,3,2,1,1,FUELHRAB002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1,PEI,2035,F_T_RDSL,2035,0.000000,2020 M$/PJ,renewable diesel,F1,2,3,2,1,1,FUELHRPEI002
0,PEI,2040,F_IMP_RDSL,2040,34.286608,2020 M$/PJ,renewable diesel,F1,2,3,2,1,1,FUELHRPEI002
1,PEI,2040,F_T_RDSL,2040,0.000000,2020 M$/PJ,renewable diesel,F1,2,3,2,1,1,FUELHRPEI002
0,PEI,2045,F_IMP_RDSL,2045,34.286608,2020 M$/PJ,renewable diesel,F1,2,3,2,1,1,FUELHRPEI002


In [41]:
data_fuel['CostVariable'] = fuels_cost_var

In [62]:
for table, df in data_fuel.items():
    if 'T_h2' in df.values:
        data_fuel[table] = df.replace('T_h2','T_h2_700')
        print(data_fuel[table].loc[data_fuel[table]['name'].str.contains('h2')])

In [64]:
mgmt.update_sqlite(db_file, data_fuel)

Inserting into MetaData with columns: element,value,notes
Inserting into MetaDataReal with columns: element,value,notes
Inserting into SeasonLabel with columns: season,notes
Inserting into SectorLabel with columns: sector,notes
Inserting into CapacityCredit with columns: region,period,tech,vintage,credit,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
Inserting into CapacityFactorProcess with columns: region,period,season,tod,tech,vintage,factor,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
Inserting into CapacityFactorTech with columns: region,period,season,tod,tech,factor,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
Inserting into CapacityToActivity with columns: region,tech,c2a,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
Inserting into Commodity with columns: name,flag,description,data_id
Inserting into CommodityType with columns: label,description
Inserting into ConstructionInput with columns: regi

In [65]:
import sqlite3
db_path = db_file
table = 'Efficiency'
column = 'output_comm'
value = 'T_h2'

try:
    # 1. Connect to the database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # 2. Execute the DELETE command
    # Use a parameterized query for best practice, though not strictly 
    query = f'DELETE FROM {table} WHERE {column} = ?'
    cursor.execute(query, (value,))

    # 3. Commit the changes and report back
    conn.commit()
    print(f"Successfully deleted {cursor.rowcount} row(s) where {column} = {value}.")

except sqlite3.Error as e:
    print(f"An error occurred: {e}")

finally:
    # 4. Always close the connection
    if conn:
        conn.close()

Successfully deleted 50 row(s) where output_comm = T_h2.
